In [ ]:
%load_ext autoreload
%autoreload 2

from edunn import utils
import edunn as nn
import numpy as np

# Squared Error Layer

In this exercise you must implement the `SquaredError` error layer, which lets you compute the error of a batch of examples. 

Error layers differ from regular layers for two reasons:

1. Their input is not only the output of the previous layer, but also the value expected from that layer (`y` and `y_true`). 
2. For a batch of $n$ examples, their output is a vector of size $n$. That is, they report the error of each example as a scalar (a real number).

The error layer must also be able to perform the `backward` operation, so that the gradient of the error can be propagated back through the network. 



# The forward method


The `forward` method of the `SquaredError` layer must simply compute the squared Euclidean distance between `y`, the values produced by the network, and `y_true`, the value expected from it.

For example, if $y=[2,-2]$ and $y_{true}=[3,3]$, then the layer's output is:

$$E(y,y_{true})=d_2(y,y_{true})=d_2([2,-2],[3,3])=(2-3)^2+(-2-3)^2 = 1^2+(-5)^2=1+25=26$$

In general, given two vectors $a=[a_1,\dots,a_n]$ and $b=[b_1,\dots,b_n]$, the squared Euclidean distance $d_2$ is:

$$
d_2(a,b)= d_2([a_1,\dots,a_n],[b_1,\dots,b_n]) =(a_1-b_1)^2+\dots+(a_n-b_n)^2
$$

For a batch of examples, the computation is independent for each example. It is therefore important that the sum of the squared differences be taken over each example (row) and not over each feature (column).


In [ ]:
y = np.array([[2,-2],
             [-4,4]])
y_true = np.array([[3,3],
             [-5,2]])


layer=nn.SquaredError()
E=np.array([[26],[5]])
utils.check_same(E,layer.forward(y_true,y))

# The backward method

Now you can compute the error of a network, great! This is the final layer of the network while it is being trained. Therefore the backward method of an error layer does not receive $\frac{δE}{δy}$; in fact, it must compute it directly from $y$, $y_{true}$, and the definition of the error. There are no parameters either. It does receive $\frac{δE}{δE_i}$, the derivative of the total error with respect to the error of each example, computed by the `MeanError` layer of the next guide.

Therefore, in this case the derivative is simple. We only need to compute $\frac{δE}{δy}$, the derivative of the error with respect to the output computed by the network, $y$.
In this case $E$ is symmetric with respect to its inputs, so let us again call them $a$ and $b$, and compute the derivative with respect to element $i$ of $a$ (the one for $b$ would be the same):

$$
\frac{δE(a,b)}{δa_i} = \frac{δ((a_1-b_1)^2+\dots+(a_n-b_n)^2)}{δa_i} \\
= \frac{δ((a_i-b_i)^2)}{δa_i} = 2 (a_i-b_i) \frac{δ((a_i-b_i))}{δa_i} \\
= 2 (a_i-b_i) 1 = 2 (a_i-b_i)
$$
Generalizing to the whole vector $a$, we get:
$$
\frac{δE(a,b)}{δa} = 2 (a-b)
$$
Where $a-b$ is a subtraction between vectors. Therefore `backward` must return:
$$
\frac{δE}{δy} = \frac{δE}{δE_i} \, 2 (y-y_{true})
$$

Again, since this error is computed per example, the calculations are independent for each row.

In [ ]:
# number of random values of x and δEδy to generate and test gradients
samples = 100
batch_size=2
features_in=3
features_out=5
input_shape=(batch_size,features_in)


layer=nn.SquaredError()
utils.check_gradient.squared_error(layer,input_shape,samples=samples)    
